### Gold Layer

In [34]:
today_date = '9999-09-09'#'2026-05-26'
workspace = "NA"

StatementMeta(, 8e69f022-ea17-476e-afec-2c697cd18f4a, 36, Finished, Available, Finished, False)

### Creating Dimension Tables

#### creating dim_student table

In [11]:
fabric_silver_path = f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/LH_Silver.Lakehouse/Tables/silver_data"

from pyspark.sql.functions import col
df  = spark.read.format('delta').load(fabric_silver_path).filter(col("Processing_Date") == str(today_date))



StatementMeta(, 8e69f022-ea17-476e-afec-2c697cd18f4a, 13, Finished, Available, Finished, False)

In [13]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from delta.tables import DeltaTable

# Define schemas for dimension tables
dim_student_schema = StructType([
    StructField("Student_ID", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Demographic_Group", StringType(), True),
    StructField("Internet_Access", StringType(), True),
    StructField("Learning_Disabilities", StringType(), True),
    StructField("Preferred_Learning_Style", StringType(), True),
    StructField("Language_Proficiency", StringType(), True),
    StructField("Parent_Involvement", StringType(), True)
])

DeltaTable.createIfNotExists(spark).tableName("Dim_Student")\
          .addColumns(dim_student_schema)\
          .execute()

StatementMeta(, 8e69f022-ea17-476e-afec-2c697cd18f4a, 15, Finished, Available, Finished, False)

#### Creating dim_course table

In [14]:
dim_course_schema = StructType([
    StructField('Course_ID', StringType(), True),
    StructField('Course_Name', StringType(), True),
    StructField('Grade_Level', StringType(), True)
])

DeltaTable.createIfNotExists(spark).tableName("Dim_Course")\
          .addColumns(dim_course_schema)\
          .execute()

StatementMeta(, 8e69f022-ea17-476e-afec-2c697cd18f4a, 16, Finished, Available, Finished, False)

### Creating Fact_Student_Performance

In [15]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

# Define schema for fact table
fact_student_performance_schema = StructType([
    StructField("Student_ID", StringType(), True),
    StructField("Course_ID", StringType(), True),
    StructField("Enrollment_Date", DateType(), True),
    StructField("Completion_Date", DateType(), True),
    StructField("Status", StringType(), False),
    StructField("Final_Grade", StringType(), False),
    StructField("Attendance_Rate", DoubleType(), False),
    StructField("Time_Spent_on_Course_hrs", DoubleType(), False),
    StructField("Assignments_Completed", IntegerType(), False),
    StructField("Quizzes_Completed", IntegerType(), False),
    StructField("Forum_Posts", IntegerType(), False),
    StructField("Messages_Sent", IntegerType(), False),
    StructField("Quiz_Average_Score", DoubleType(), False),
    StructField("Assignment_Scores", StringType(), True),
    StructField("Assignment_Average_Score", DoubleType(), False),
    StructField("Project_Score", DoubleType(), False),
    StructField("Extra_Credit", DoubleType(), False),
    StructField("Overall_Performance", DoubleType(), False),
    StructField("Feedback_Score", DoubleType(), False),
    StructField("Completion_Time_Days", IntegerType(), True),
    StructField("Performance_Score", DoubleType(), False),
    StructField("Course_Completion_Rate", StringType(), False),
    StructField("Processing_Date", DateType(), True)
])

DeltaTable.createIfNotExists(spark).tableName("Fact_Student_Performance")\
          .addColumns(fact_student_performance_schema)\
          .execute()

StatementMeta(, 8e69f022-ea17-476e-afec-2c697cd18f4a, 17, Finished, Available, Finished, False)

### Loading Data into Dim_Student

In [20]:
df_selected_dim_student = (df.select( "Student_ID", 
    "Name", 
    "Age", 
    "Gender", 
    "Demographic_Group", 
    "Internet_Access", 
    "Learning_Disabilities", 
    "Preferred_Learning_Style", 
    "Language_Proficiency", 
    "Parent_Involvement"))

StatementMeta(, 8e69f022-ea17-476e-afec-2c697cd18f4a, 22, Finished, Available, Finished, False)

In [21]:
dim_student_table_path = f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/LH_Gold.Lakehouse/Tables/dim_student"
dim_student_table = DeltaTable.forPath(spark, dim_student_table_path)

dim_student_table.alias("target").merge(
    df_selected_dim_student.alias("source"),
    "target.Student_ID =source.Student_ID"
).whenMatchedUpdate(set = {
    "Name": "source.Name", 
    "Age": "source.Age",
    "Gender": "source.Gender", 
    "Demographic_Group": "source.Demographic_Group",
    "Internet_Access": "source.Internet_Access",
    "Learning_Disabilities": "source.Learning_Disabilities", 
    "Preferred_Learning_Style":  "source.Preferred_Learning_Style",
    "Language_Proficiency":  "source.Language_Proficiency",
    "Parent_Involvement":  "source.Parent_Involvement"
}).whenNotMatchedInsert(values = {
    "Student_ID": "source.Student_ID",
    "Name": "source.Name",
    "Age": "source.Age",
    "Gender": "source.Gender",
    "Demographic_Group": "source.Demographic_Group",
    "Internet_Access": "source.Internet_Access",
    "Learning_Disabilities": "source.Learning_Disabilities",
    "Preferred_Learning_Style": "source.Preferred_Learning_Style",
    "Language_Proficiency": "source.Language_Proficiency",
    "Parent_Involvement": "source.Parent_Involvement"

}).execute()

StatementMeta(, 8e69f022-ea17-476e-afec-2c697cd18f4a, 23, Finished, Available, Finished, False)

In [23]:
# Get the history of the Delta table to extract metrics
history_df = dim_student_table.history(1)  # Get the latest operation

# Extract metrics from the history DataFrame
operation_metrics = history_df.select("operationMetrics").collect()[0][0]

# Extract specific metrics
rows_inserted = operation_metrics.get('numTargetRowsInserted', 0)
rows_updated = operation_metrics.get('numTargetRowsUpdated', 0)
rows_deleted = operation_metrics.get('numTargetRowsDeleted', 0)
rows_affected = int(rows_inserted) + int(rows_updated) + int(rows_deleted) 

print('Total rows of table: ',dim_student_table.toDF().count())
print("Merge Metrics:")
print(f"Rows inserted: {rows_inserted}")
print(f"Rows updated: {rows_updated}")
print(f"Rows deleted: {rows_deleted}")
print(f"Total rows affected: {rows_affected}")

StatementMeta(, 8e69f022-ea17-476e-afec-2c697cd18f4a, 25, Finished, Available, Finished, False)

Total rows of table:  25
Merge Metrics:
Rows inserted: 0
Rows updated: 25
Rows deleted: 0
Total rows affected: 25


### Loading Data into Dim_Course table

In [25]:
# Selecting required column for Dim_course
df_selected_dim_course = (df.select( "Course_ID",
                                     "Course_Name",
                                     "Grade_Level"
                                    )
                         )

StatementMeta(, 8e69f022-ea17-476e-afec-2c697cd18f4a, 27, Finished, Available, Finished, False)

In [26]:
from delta.tables import *

dim_course_table_path = f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/LH_gold.Lakehouse/Tables/dim_course"
dim_course_table = DeltaTable.forPath(spark,dim_course_table_path)

# Perform the MERGE (UPSERT) operation using PySpark Syntax

dim_course_table.alias("target").merge(
    df_selected_dim_course.alias("source"),
    "target.Course_ID = source.Course_ID"
).whenMatchedUpdate(set={
    "Course_ID": "source.Course_ID",
    "Course_Name": "source.Course_Name",
    "Grade_Level": "source.Grade_Level"
}).whenNotMatchedInsert(values={
     "Course_ID": "source.Course_ID",
    "Course_Name": "source.Course_Name",
    "Grade_Level": "source.Grade_Level"
}).execute()


StatementMeta(, 8e69f022-ea17-476e-afec-2c697cd18f4a, 28, Finished, Available, Finished, False)

In [27]:
history_df = dim_course_table.history(1)  # Get the latest operation

# Extract metrics from the history DataFrame
operation_metrics = history_df.select("operationMetrics").collect()[0][0]

# Extract specific metrics
rows_inserted = operation_metrics.get('numTargetRowsInserted', 0)
rows_updated = operation_metrics.get('numTargetRowsUpdated', 0)
rows_deleted = operation_metrics.get('numTargetRowsDeleted', 0)
rows_affected = int(rows_inserted) + int(rows_updated) + int(rows_deleted)

print('Total rows of table: ',dim_course_table.toDF().count())
print("Merge Metrics:")
print(f"Rows inserted: {rows_inserted}")
print(f"Rows updated: {rows_updated}")
print(f"Rows deleted: {rows_deleted}")
print(f"Total rows affected: {rows_affected}")

StatementMeta(, 8e69f022-ea17-476e-afec-2c697cd18f4a, 29, Finished, Available, Finished, False)

Total rows of table:  25
Merge Metrics:
Rows inserted: 25
Rows updated: 0
Rows deleted: 0
Total rows affected: 25


### Loading Data Into Fact_Student_Performance

In [28]:
df_selected_Fact_student = (df.select( 
    "Student_ID",
    "Course_ID",
    "Enrollment_Date",
    "Completion_Date",
    "Status",
    "Final_Grade",
    "Attendance_Rate",
    "Time_Spent_on_Course_hrs",
    "Assignments_Completed",
    "Quizzes_Completed",
    "Forum_Posts",
    "Messages_Sent",
    "Quiz_Average_Score",
    "Assignment_Scores",
    "Assignment_Average_Score",
    "Project_Score",
    "Extra_Credit",
    "Overall_Performance",
    "Feedback_Score",
    "Completion_Time_Days",
    "Performance_Score",
    "Course_Completion_Rate",
    "Processing_Date"
))

StatementMeta(, 8e69f022-ea17-476e-afec-2c697cd18f4a, 30, Finished, Available, Finished, False)

In [29]:
from delta.tables import *

fact_student_table_path = f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/LH_gold.Lakehouse/Tables/fact_student_performance"
fact_student_table = DeltaTable.forPath(spark,fact_student_table_path)

# Perform the MERGE (UPSERT) operation and capture the operation metrics
merge_operation = fact_student_table.alias("target").merge(
    df_selected_Fact_student.alias("source"),
    "target.Student_ID = source.Student_ID AND target.Course_ID = source.Course_ID"
).whenMatchedUpdate(set={
    "Student_ID": "source.Student_ID",
    "Course_ID": "source.Course_ID",
    "Enrollment_Date": "source.Enrollment_Date",
    "Completion_Date": "source.Completion_Date",
    "Status": "source.Status",
    "Final_Grade": "source.Final_Grade",
    "Attendance_Rate": "source.Attendance_Rate",
    "Time_Spent_on_Course_hrs": "source.Time_Spent_on_Course_hrs",
    "Assignments_Completed": "source.Assignments_Completed",
    "Quizzes_Completed": "source.Quizzes_Completed",
    "Forum_Posts": "source.Forum_Posts",
    "Messages_Sent": "source.Messages_Sent",
    "Quiz_Average_Score": "source.Quiz_Average_Score",
    "Assignment_Scores": "source.Assignment_Scores",
    "Assignment_Average_Score": "source.Assignment_Average_Score",
    "Project_Score": "source.Project_Score",
    "Extra_Credit": "source.Extra_Credit",
    "Overall_Performance": "source.Overall_Performance",
    "Feedback_Score": "source.Feedback_Score",
    "Completion_Time_Days": "source.Completion_Time_Days",
    "Performance_Score": "source.Performance_Score",
    "Course_Completion_Rate": "source.Course_Completion_Rate",
    "Processing_Date": "source.Processing_Date"
}).whenNotMatchedInsert(values={
    "Student_ID": "source.Student_ID",
    "Course_ID": "source.Course_ID",
    "Enrollment_Date": "source.Enrollment_Date",
    "Completion_Date": "source.Completion_Date",
    "Status": "source.Status",
    "Final_Grade": "source.Final_Grade",
    "Attendance_Rate": "source.Attendance_Rate",
    "Time_Spent_on_Course_hrs": "source.Time_Spent_on_Course_hrs",
    "Assignments_Completed": "source.Assignments_Completed",
    "Quizzes_Completed": "source.Quizzes_Completed",
    "Forum_Posts": "source.Forum_Posts",
    "Messages_Sent": "source.Messages_Sent",
    "Quiz_Average_Score": "source.Quiz_Average_Score",
    "Assignment_Scores": "source.Assignment_Scores",
    "Assignment_Average_Score": "source.Assignment_Average_Score",
    "Project_Score": "source.Project_Score",
    "Extra_Credit": "source.Extra_Credit",
    "Overall_Performance": "source.Overall_Performance",
    "Feedback_Score": "source.Feedback_Score",
    "Completion_Time_Days": "source.Completion_Time_Days",
    "Performance_Score": "source.Performance_Score",
    "Course_Completion_Rate": "source.Course_Completion_Rate",
    "Processing_Date": "source.Processing_Date"
}).execute()

StatementMeta(, 8e69f022-ea17-476e-afec-2c697cd18f4a, 31, Finished, Available, Finished, False)

In [30]:
## History 

# Get the history of the Delta table to extract metrics
history_df = fact_student_table.history(1)  # Get the latest operation

# Extract metrics from the history DataFrame
operation_metrics = history_df.select("operationMetrics").collect()[0][0]

# Extract specific metrics
rows_inserted = operation_metrics.get('numTargetRowsInserted', 0)
rows_updated = operation_metrics.get('numTargetRowsUpdated', 0)
rows_deleted = operation_metrics.get('numTargetRowsDeleted', 0)
rows_copied = operation_metrics.get('numTargetRowsCopied', 0)
rows_affected = int(rows_inserted) + int(rows_updated) + int(rows_deleted) + int(rows_copied)

print('Total rows of table: ',fact_student_table.toDF().count())
print("Merge Metrics:")
print(f"Rows inserted: {rows_inserted}")
print(f"Rows updated: {rows_updated}")
print(f"Rows deleted: {rows_deleted}")
#print(f"Rows copied: {rows_copied}")
print(f"Total rows affected: {rows_affected}")

StatementMeta(, 8e69f022-ea17-476e-afec-2c697cd18f4a, 32, Finished, Available, Finished, False)

Total rows of table:  25
Merge Metrics:
Rows inserted: 25
Rows updated: 0
Rows deleted: 0
Total rows affected: 25
